## Recommended Starting Hyperparameters
1. Population Size
* Typical Range: 50–500.
* Recommendation: Start with 100–200.
* Larger populations explore more, but are slower per generation.
* With your large search space, erring toward the higher end (e.g., 200) is reasonable if your fitness function is fast.

2. Number of Generations
* Typical Range: 50–500+.
* Recommendation: 100–300 generations.
* More generations allow more refinement, but diminishing returns after a point.
* You can monitor if the best fitness plateaus early.
3. Selection Method
* Tournament selection or roulette wheel selection are both good.
* Many libraries use tournament selection by default.
4. Crossover Rate
* Typical: 0.7–0.9 (i.e., 70–90% of offspring are produced by crossover).
* Recommendation: 0.8 (80%).
5. Mutation Rate
* Typical: 0.01–0.1 (1–10% mutation rate per gene).
* Recommendation: 0.05 (5% per gene).
* With 9 genes (slots), this means about half your children will have at least one mutation per generation.
6. Elitism
* Keep the best 1–2 individuals from each generation to ensure you don’t lose the best found so far.


In [1]:
import pandas as pd
from itertools import chain
from functools import reduce

In [2]:
# Hyperparameters
POPULATION_SIZE = 150
NUM_GENERATIONS = 100
MUTATION_RATE = 0.05
EXCLUSIONS = {
    "m_flags" : [3208,3080]
}
LOWER_BOUNDS = {
    # "level" : 190
}
UPPER_BOUNDS = {}

### Load

In [3]:
def load_csv(file_path):
    """Load a CSV file into a DataFrame."""
    df = pd.read_csv(file_path,index_col=0,
                     dtype={k:str for k in ["id", "nameId", "typeId", "set","effectId","characteristic"]})
    df.index = df.index.astype(str)
    return df

In [4]:
items = load_csv("data/datasets/items.csv")
item_sets = load_csv("data/datasets/item_sets.csv")
set_effects = load_csv("data/datasets/set_effects.csv")
effects = load_csv("data/datasets/effects.csv").set_index("id")
item_types = load_csv("data/datasets/item_types.csv").set_index("id").astype(str)

/var/folders/c3/9yq4kl5j7qd715s6mq5tyslm0000gp/T/ipykernel_13177/269983047.py:3: DtypeWarning: Columns (14,15,19,20,21,22,23,24,27,28,29,30,32,33,63,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path,index_col=0,


### Filters

In [5]:
for k,v in EXCLUSIONS.items():
    items = items[~items[k].isin(v)]

In [6]:
remove_list = []
for k,v in LOWER_BOUNDS.items():
    remove_list += items[items["type"]!="EffectInstanceDice"][items[k] < v].index.to_list()

for k,v in UPPER_BOUNDS.items():
    remove_list += items[items["type"]!="EffectInstanceDice"][items[k] > v].index.to_list()

items = items.drop(remove_list)

In [7]:
allowed_typeIds = list([str(i) for i in chain(range(1,12), range(16,22))])
weapon_typeIds = list(set(items[items["type"]=="Weapons"]["typeId"].to_list()))
normal_typeIds = [i for i in allowed_typeIds if i not in weapon_typeIds and i != "9"]
extra_typeIds = ["23","151","217"]

In [8]:
EXTRA_SLOTS = 6
pools = [items[items["typeId"].isin(weapon_typeIds)].index.to_list()]\
    + [items[items["typeId"].isin([i])].index.to_list() for i in normal_typeIds]
for _ in range(2): # Add 2 slots for rings
    pools += [items[items["typeId"].isin(["9"])].index.to_list()]
for _ in range(EXTRA_SLOTS):
    pools += [items[items["typeId"].isin(extra_typeIds)].index.to_list()]
NUM_TYPES = len(pools)

## Effect logic

In [9]:
def get_item_effects(item):
    if isinstance(item["possibleEffects"], float) or not item["possibleEffects"]:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    # Get the effects of the item
    item_effects = items.loc[[i for i in item["possibleEffects"].split(",")]]
    # Filter effects to only include those that are in the effects DataFrame
    return item_effects[item_effects["effectId"].isin(effects.index)]

In [10]:
def get_item_contribution(item):
    # Get the effects of the item
    item_effects = get_item_effects(item)
    if item_effects.empty:
        return pd.Series(dtype=float)
    # Merge with effects DataFrame to get characteristics and operators
    item_effects["characteristic"] = effects.loc[item_effects["effectId"]]["characteristic"].values
    item_effects["characteristicOperator"] = effects.loc[item_effects["effectId"]]["characteristicOperator"].apply(lambda x: 1 if x == "+" else -1).values
    # Calculate the maximum value for each effect based on diceSide and diceNum (maximum of the two)
    item_effects["effectValue"] = item_effects["characteristicOperator"] * item_effects[["diceSide", "diceNum"]].max(axis=1)
    # Return a DataFrame with the characteristic and effectValue
    return item_effects[["characteristic", "effectValue"]].set_index("characteristic")["effectValue"].groupby("characteristic").sum()

contributions = {rid: get_item_contribution(items.loc[rid]) for rid in items.index}

In [11]:
def get_set_contribution(item_set,overlap):
    if overlap == 0:
        return pd.Series(dtype=float)
    if item_set.name not in set_effects.index:
        # If the set has no effects, return an empty Series
        return pd.Series(dtype=float)
    # Get the effects of the set based on the overlap
    s_effects = set_effects.loc[item_set.name][str(overlap)]
    if pd.isna(s_effects) or not s_effects:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    s_effects = s_effects.split(",")
    if "-2" in s_effects:
        # Remove the "-2" from the list of effects
        s_effects.remove("-2")
    s_effects = item_sets.loc[s_effects]
    if s_effects.empty:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    # Merge with effects DataFrame to get characteristics and operators
    s_effects["characteristic"] = effects.loc[s_effects["effectId"]]["characteristic"].values
    s_effects["characteristicOperator"] = effects.loc[s_effects["effectId"]]["characteristicOperator"].apply(lambda x: 1 if x == "+" else -1).values
    # Calculate the maximum value for each effect based on diceSide and diceNum (maximum of the two)
    s_effects["effectValue"] = s_effects["characteristicOperator"] * s_effects[["diceSide", "diceNum"]].max(axis=1)
    # Return a DataFrame with the characteristic and effectValue
    return s_effects[["characteristic", "effectValue"]].set_index("characteristic")["effectValue"].groupby("characteristic").sum()


In [12]:
set_contributions = {}
for rid,row in item_sets.iterrows():
    if isinstance(row["items"], float) or not row["items"]:
        # If there are no items, return an empty Series
        continue
    set_contributions[rid] = {}
    size = len(row["items"].split(","))
    for i in range(size):
        set_contributions[rid][i+1] = get_set_contribution(row, i+1)
    

## 0. Methods

In [13]:
import random
def random_chromosome():
    return [random.randint(0, len(slot_pool)-1) for slot_pool in pools]

In [14]:
# Mutation
def mutate(chromosome, mutation_rate=MUTATION_RATE):
    # Create a new chromosome by mutating some slots
    new_chromosome = chromosome[:]
    for i, slot_items in enumerate(pools):
        if random.random() < mutation_rate:
            new_chromosome[i] = random.randint(0, len(slot_items)-1)
    return new_chromosome

In [15]:
# Crossover
def crossover(parent1, parent2):
    # Perform single-point crossover
    point = random.randint(1, NUM_TYPES-2)
    return parent1[:point] + parent2[point:]

In [16]:
# # Sanity check to make sure the function works on all items
# for row,item in items.iterrows():
#     get_item_contribution(item)

In [17]:
# Fitness function
def get_totals(chromosome):
    chromosome_items = [items.loc[pools[idx][item]].name for idx,item in enumerate(chromosome)]
    # Item effects
    lst = [contributions[item] for item in chromosome_items]

    # Set effects
    item_ids = items.loc[chromosome_items]["id"].values
    associated_sets = items.loc[chromosome_items]["set"].dropna().unique()
    for set_rid,set_row in item_sets.loc[associated_sets].iterrows():
        set_items = set_row["items"].split(",")
        overlap = len(set(set_items) & set(item_ids))
        lst.append(set_contributions[set_rid][overlap])
        
    # Totals
    totals = reduce(lambda x, y: x.add(y, fill_value=0), lst).to_dict()
    return totals

def get_damage(weapon, chromosome):
    totals = get_totals(chromosome)
    weapon_effects = get_item_effects(weapon)
    if weapon_effects.empty:
        return 0
    damage_mapper = {
            "98" : "14", # Air,
            "96" : "13", # Water
            "99" : "15", # Fire
            "97" : "10", # Earth
            "100" : "10" # Neutral (by earth)
        }

    bonus_damage_mapper = {
        "98" : "91", # Air,
        "96" : "90", # Water
        "99" : "89", # Fire
        "97" : "88", # Earth
        "100" : "92", # Neutral
    }
    
    # Character stats
    power = totals.get("25",0)
    base_crit_chance = totals.get("18",0)
    base_crit_added_damage = totals.get("86",0)

    # Calculate damage
    weapon_damage = weapon_effects[weapon_effects["effectId"].isin(damage_mapper)]

    # Critical hit logic
    crit_bonus = weapon["criticalHitBonus"] if not pd.isna(weapon["criticalHitBonus"]) else 0
    weapon_crit_chance = weapon["criticalHitProbability"] if not pd.isna(weapon["criticalHitProbability"]) else 0
    crit_chance = min(weapon_crit_chance + base_crit_chance,100)
    expected_crit_bonus = crit_bonus * crit_chance/100 # Expected value of distribution
    expected_crit_added_damage = base_crit_added_damage * crit_chance/100 # Expected value of distribution
    # max_crit_bonus = crit_bonus # Use this for maximum possible


    # Calculate total damage
    damage = 0
    for _,row in weapon_damage.iterrows():
        # Get the stat and base damage
        stat = damage_mapper[row["effectId"]]
        weapon_base_damage = row[["diceNum","diceSide"]].max()
        stat_base = totals.get(stat,0)

        # Get the added bonus damage from the stat
        bonus_damage_id = bonus_damage_mapper[row["effectId"]]
        added_bonus = totals.get(bonus_damage_id, 0)
        damage += (weapon_base_damage + expected_crit_bonus) * (1+(stat_base+power)/100) + expected_crit_added_damage + added_bonus
    return damage
    

In [18]:
# weapon = items.loc["2861093746004724544"]
# chromosome = random_chromosome()
# weapon = get_chromosome_weapon(chromosome)
# totals = get_totals(chromosome)
# get_damage(weapon, chromosome)
# items.loc[[pools[idx][item] for idx,item in enumerate(chromosome)]]
# get_item_effects(weapon)[["name","effectId","diceNum","diceSide"]]


In [19]:
def get_chromosome_weapon(chromosome):
    chromosome_item_ids = [items.loc[pools[idx][item]].name for idx,item in enumerate(chromosome)]
    equipped_items = items.loc[chromosome_item_ids]
    return equipped_items[equipped_items["type"]=="Weapons"].iloc[0]

In [20]:
def fitness(chromosome):
    # Penalizations
    if len(set(chromosome[-EXTRA_SLOTS:])) < EXTRA_SLOTS:
        # Some dofus or trophies are duplicated
        return 0
    # return get_totals(chromosome).get(18, 0)
    weapon = get_chromosome_weapon(chromosome)
    return get_damage(weapon,chromosome)

In [21]:
# Sanity check to make sure the function works on a random chromosome
chromosome = random_chromosome()
assert isinstance(fitness(chromosome), float) or isinstance(fitness(chromosome), int)

## 1. Initialize population

In [22]:
population = [random_chromosome() for _ in range(POPULATION_SIZE)]

## 2. Selection

In [23]:
def select(population, k=10):
    # Select top k chromosomes
    scored = sorted(population, key=fitness, reverse=True)
    return scored[:k]

## 3. Main Loop

In [ ]:
for gen in range(NUM_GENERATIONS):
    # Select best individuals
    selected = select(population, k=10)
    # Generate new population
    new_population = selected[:]
    while len(new_population) < POPULATION_SIZE:
        parent1, parent2 = random.sample(selected, 2)
        child = crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)
    population = new_population
    # Print best fitness in this generation
    best = max(population, key=fitness)
    print(f'Gen {gen}: Best damage = {fitness(best)}')

Gen 0: Best damage = 632.6750000000001
Gen 1: Best damage = 732.85
Gen 2: Best damage = 783.8499999999999
Gen 3: Best damage = 845.5749999999999
Gen 4: Best damage = 896.575
Gen 5: Best damage = 927.5074999999999
Gen 6: Best damage = 984.095
Gen 7: Best damage = 1045.645
Gen 8: Best damage = 1082.545
Gen 9: Best damage = 1111.7450000000001
Gen 10: Best damage = 1111.7450000000001
Gen 11: Best damage = 1139.42


In [ ]:
items.loc[[pools[idx][item] for idx,item in enumerate(best)]]

,name,type,m_flags,id,nameId,typeId,descriptionId,iconId,level,realWeight,...,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero,class,set
2861093746004724543,Hoz maldita del gran Desangrador Guerrero,Weapons,24584,8992,224887,22,306152.0,22003.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Guadaña,NaN
2861093746004736647,Amuleto de Wulán,Items,24712,24029,879254,1,879255.0,1317.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Amuleto,2861093746001838376
2861093746004729343,Alianza de levitrof,Items,24712,15193,419341,9,419342.0,9290.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anillo,2861093746001838260
2861093746004729192,Cinturón de Papa No-es,Items,24712,15009,375986,10,375987.0,10263.0,198.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cinturón,2861093746001838252
2861093746004729193,Botas de Papa No-es,Items,24712,15010,373954,11,373955.0,11255.0,197.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bota,2861093746001838252
2861093746004729342,Sombrero de levitrof,Items,24712,15192,419339,16,419340.0,16413.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sombrero,2861093746001838260
2861093746004727614,Capa Brigada,Items,24712,13197,242230,17,242231.0,17254.0,198.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Capa,NaN
2861093746004728051,Lokulto,Items,24712,13673,241771,18,845905.0,18091.0,20.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Mascota,NaN
2861093746001847005,Dofus Púrpura,Items,16520,694,39487,23,39488.0,23001.0,110.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dofus,NaN
2861093746004730368,Lozano mayor,Items,24712,16332,515826,151,515827.0,151204.0,150.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Trofeo,NaN


In [ ]:
fitness(best)

np.float64(1432.24)

In [ ]:
item_sets.loc[tmp["set"].dropna()]

,name,type,id,items,nameId,bonusIsSecret,effects,m_flags,effectUid,baseEffectId,...,delay,triggers,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero
2861093746001838376,Set de Wulán,ItemSets,491,"24029,24030,24031",876499,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001838260,Set de levitrof,ItemSets,333,"15192,15193,15194",409184,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001838252,Set de Papa No-es,ItemSets,323,"15009,15010,15011",372650,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001838252,Set de Papa No-es,ItemSets,323,"15009,15010,15011",372650,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2861093746001838260,Set de levitrof,ItemSets,333,"15192,15193,15194",409184,0.0,{'Array': [{'values': {'Array': [{'rid': -2}]}...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Print total stats
totals = get_totals(best)
for k,v in totals.items():
    print(effects[effects["characteristic"]==k]["name"][0],v)

nan -14900.0
-#1{{~1~2 a -}}#2 PA 1.0
#1{{~1~2 a }}#2 fuerza 310.0
#1{{~1~2 a }}#2 vitalidad 1240.0
#1{{~1~2 a }}#2 sabiduría 270.0
#1{{~1~2 a }}#2 suerte 310.0
#1{{~1~2 a }}#2 agilidad 130.0
#1{{~1~2 a }}#2 inteligencia 250.0
#1{{~1~2 a }}#2 % crítico 37.0
-#1{{~1~2 a -}}#2 alcance 4.0
-#1{{~1~2 -}}#2 PM 1.0
#1{{~1~2 a }}#2 potencia (daños generales) 390.0
#1{{~1~2 a }}#2 invocaciones{{~p}}{{~z}} 2.0
#1{{~1~2 a }}#2 a la esquiva pérdidas de PA 22.0
#1{{~1~2 a }}#2% resistencia a la tierra 11.0
#1{{~1~2 a }}#2% resistencia al fuego 17.0
#1{{~1~2 a }}#2% resistencia al agua 18.0
#1{{~1~2 a }}#2% resistencia al aire 22.0
#1{{~1~2 a }}#2% resistencia neutral 5.0
#1{{~1~2 a }}#2 iniciativa 1300.0
#1{{~1~2 a }}#2 prospección 45.0
#1{{~1~2 a }}#2 huida -1.0
#1{{~1~2 a }}#2 placaje 15.0
#1{{~1~2 a }}#2 al retiro de PA -15.0
#1{{~1~2 a }}#2 al retiro de PM 6.0
#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} de empuje 20.0
#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} crítico 60.0
#1{{~1~2 a }}#2 de resistencia{{~p}

/var/folders/c3/9yq4kl5j7qd715s6mq5tyslm0000gp/T/ipykernel_10883/2902173380.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(effects[effects["characteristic"]==k]["name"][0],v)
